Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [ ]:
# Course setup for the OpenAI Agents SDK. In Colab: add the course key under
# the key icon (left sidebar) as COURSE_API_KEY.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "openai-agents", "python-dotenv"], check=True)
    from google.colab import userdata
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    KEY = os.getenv("COURSE_API_KEY") or userdata.get("COURSE_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"

from openai import AsyncOpenAI
from agents import (set_default_openai_client, set_default_openai_api,
                    set_tracing_disabled)

set_default_openai_client(AsyncOpenAI(base_url=BASE, api_key=KEY))
set_default_openai_api("chat_completions")   # no /v1/responses on these endpoints
set_tracing_disabled(True)
print(f"model={MODEL}")

### Minimal Agent OpenAI
The OpenAI Agents SDK is a lightweight, Python-first framework for building agentic applications from a small set of composable primitives, and it's the production-ready successor to OpenAI's experimental Swarm project. Its core building block is the Agent, an LLM configured with instructions, tools, and optional handoffs, guardrails, and structured outputs. A Runner drives the built-in agent loop: it sends messages to the model, executes any tool calls, feeds the results back, and iterates until the agent produces a final output (Runner.run, run_sync, or run_streamed). Function tools turn any Python function into a tool with automatic schema generation and Pydantic validation, alongside hosted tools (web search, file search, code interpreter, computer use) and MCP servers.

In [ ]:
import asyncio
from agents import Agent, Runner

# 1. Define the agent
agent = Agent(
    name="Assistant",
    instructions="You are a helpful assistant. Answer concisely.",
    model=MODEL
)

# 2. Run the agent
result = await Runner.run(agent, "What is a purchase requisition?")
print(result.final_output)



### Streaming output OpenAI
Streaming output means the model's response is delivered incrementally, piece by piece as it's produced, instead of the caller waiting for the whole thing and receiving it in one block at the end. That familiar "typing" effect where text appears word by word is streaming; the alternative is a blank screen until the full answer lands at once.
The reason it matters is partly UX and partly, for agents, visibility. A single LLM call might take several seconds and a multi-step agent thirty or more; without streaming the user stares at nothing until it's all done. Streaming cuts perceived latency (the first words appear almost immediately) and, for an agent, lets us show that it is working, calling a tool, getting a result, reasoning again, rather than appearing frozen.

In [ ]:
import asyncio
from agents import Agent, Runner
from openai.types.responses import ResponseTextDeltaEvent

agent = Agent(
    model=MODEL,
    name="Assistant",
    instructions="You are a helpful assistant.",
)

result = Runner.run_streamed(agent, input="Write three sentences explaining our expense policy to a new joiner.")
    
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)
    
print()

### Enable Tracing 
Enable_verbose_stdout_logging() is the SDK's quick debug switch : it dumps detailed logs of what the SDK is doing straight to our console (stdout), so we can see the run unfold in the terminal. It's the "just show me everything, right here, right now" option.

In [4]:
from agents import Agent, Runner, logging

# Enable detailed console logging
logger = logging.getLogger("openai.agents")
logger.handlers.clear()          # drop any handlers added by previous runs

agent = Agent(
    name="Debug Agent",
    instructions="Be helpful.",
    model=MODEL
)

# 2. Run the agent
result = await Runner.run(agent, "Hello!")
print(result.final_output)

Tracing is disabled. Not creating trace Agent workflow
Setting current trace: no-op
Tracing is disabled. Not creating span <agents.tracing.span_data.TaskSpanData object at 0x00000216FCAD89B0>
Tracing is disabled. Not creating span <agents.tracing.span_data.AgentSpanData object at 0x00000216FD3C6BD0>
Running agent Debug Agent (turn 1)
Tracing is disabled. Not creating span <agents.tracing.span_data.TurnSpanData object at 0x00000216FE489220>
No conversation_id available for request
Tracing is disabled. Not creating span <agents.tracing.span_data.GenerationSpanData object at 0x00000216F8461EB0>
Calling LLM
Received model response
Processing output item type=message class=ResponseOutputMessage
Resetting current trace
Hello! How can I help you today?


### Weather Agent OpenAI
@function_tool is the OpenAI Agents SDK decorator that turns any ordinary Python function into a tool the agent can call. The docstring becomes the tool description. That """Get the current weather for a city.""" is what the model reads to decide when to call the tool.

In [5]:
import asyncio
from agents import Agent, Runner, function_tool

# Enable detailed console logging
logger = logging.getLogger("openai.agents")
logger.handlers.clear()          # drop any handlers added by previous runs

@function_tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    data = {
        "Amsterdam": "15°C, partly cloudy",
        "London": "12°C, rainy",
        "Paris": "18°C, sunny",
        "Berlin": "10°C, overcast"
    }
    return data.get(city, f"No data for {city}")

@function_tool
def get_forecast(city: str, days: int) -> str:
    """Get a weather forecast for the next N days."""
    return f"Forecast for {city}: {days} days of mild temperatures expected, with some rain on day 2."

agent = Agent(
    name="Weather Bot",
    instructions="You are a helpful weather assistant. Use tools to get weather data.",
    model=MODEL,
    tools=[get_weather, get_forecast]
)

questions = [
    "What's the weather like in Amsterdam?",
    "Should I pack an umbrella for a trip to London tomorrow?",
    "Compare the weather in Paris and Berlin right now."
]
for q in questions:
    print(f"\nQ: {q}")
    result = await Runner.run(agent, q)
    print(f"A: {result.final_output}")


Q: What's the weather like in Amsterdam?
Tracing is disabled. Not creating trace Agent workflow
Setting current trace: no-op
Tracing is disabled. Not creating span <agents.tracing.span_data.TaskSpanData object at 0x00000216FD412760>
Tracing is disabled. Not creating span <agents.tracing.span_data.AgentSpanData object at 0x00000216FD3C7350>
Running agent Weather Bot (turn 1)
Tracing is disabled. Not creating span <agents.tracing.span_data.TurnSpanData object at 0x00000216FCAD8F50>
No conversation_id available for request
Tracing is disabled. Not creating span <agents.tracing.span_data.GenerationSpanData object at 0x00000216F8461EB0>
Calling LLM
Received model response
Processing output item type=function_call class=ResponseFunctionToolCall
Tracing is disabled. Not creating span <agents.tracing.span_data.FunctionSpanData object at 0x00000216FE5E2FD0>
Invoking tool get_weather
Tool get_weather completed.
Running agent Weather Bot (turn 2)
Tracing is disabled. Not creating span <agents.tr

### Calculator Agent OpenAI
Calculator agent with calculation functions

In [8]:
# demo_calculator.py
import asyncio
from agents import Agent, Runner, function_tool

# Enable detailed console logging
logger = logging.getLogger("openai.agents")
logger.handlers.clear()          # drop any handlers added by previous runs

@function_tool
def add(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b

@function_tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together."""
    return a * b

@function_tool
def power(base: float, exponent: float) -> float:
    """Raise base to the power of exponent."""
    return base ** exponent

agent = Agent(
    name="Math Agent",
    instructions="You are a math assistant. Use tools for all calculations.",
    model=MODEL,
    tools=[add, multiply, power]
)

result = await Runner.run(
    agent,
    "What is ((1 + 4) * 2) to the power of 3?"
)
print("\nFinal answer:", result.final_output)

Tracing is disabled. Not creating trace Agent workflow
Setting current trace: no-op
Tracing is disabled. Not creating span <agents.tracing.span_data.TaskSpanData object at 0x00000216FE68C500>
Tracing is disabled. Not creating span <agents.tracing.span_data.AgentSpanData object at 0x00000216FD443290>
Running agent Math Agent (turn 1)
Tracing is disabled. Not creating span <agents.tracing.span_data.TurnSpanData object at 0x00000216FE68C190>
No conversation_id available for request
Tracing is disabled. Not creating span <agents.tracing.span_data.GenerationSpanData object at 0x00000216FD4434D0>
Calling LLM
Received model response
Processing output item type=message class=ResponseOutputMessage
Processing output item type=function_call class=ResponseFunctionToolCall
Tracing is disabled. Not creating span <agents.tracing.span_data.FunctionSpanData object at 0x00000216FE68C370>
Invoking tool add
Tool add completed.
Running agent Math Agent (turn 2)
Tracing is disabled. Not creating span <agent

### Chat with Memory OpenAI
In this code, history is handled manually : we are maintaining the conversation ourselves in that history list, rather than letting the SDK do it. Let me trace exactly how it flows, because the mechanism is the whole point.
The history list is a plain list of message dicts ({"role": ..., "content": ...}) that we grow by hand across turns. Each loop iteration does three things in order:

Append the user's message : history.append({"role": "user", "content": user_input}) adds what they just typed.
Send the entire history to the agent : Runner.run(agent, history). This is the key line: we pass the whole list, every earlier turn included. The agent sees every prior turn, which is what gives it "memory" of the conversation.
Append the assistant's reply : history.append({"role": "assistant", "content": assistant_reply}) records the answer so it's present on the next turn

### Using History OpenAI
Following demonstrates of how to chain turns so the agent remembers across a conversation.

In [10]:
import asyncio
from agents import Agent, Runner
import logging

# Enable detailed console logging
logger = logging.getLogger("openai.agents")
logger.handlers.clear()          # drop any handlers added by previous runs

agent = Agent(
    name="Memory Agent",
    instructions="Remember everything the user tells you.",
    model=MODEL
)

# Turn 1
result1 = await Runner.run(agent, "My favorite color is blue and I love Python.")
print("Turn 1:", result1.final_output)
    
# Turn 2 – use result1 as context for next turn
history = result1.to_input_list()
history.append({"role": "user", "content": "What's my favorite color?"})
    
result2 = await Runner.run(agent, history)
print("Turn 2:", result2.final_output)
    
# Turn 3 – chain again
history2 = result2.to_input_list()
history2.append({"role": "user", "content": "What language do I love?"})
    
result3 = await Runner.run(agent, history2)
print("Turn 3:", result3.final_output)

Tracing is disabled. Not creating trace Agent workflow
Setting current trace: no-op
Tracing is disabled. Not creating span <agents.tracing.span_data.TaskSpanData object at 0x00000216FE68C640>
Tracing is disabled. Not creating span <agents.tracing.span_data.AgentSpanData object at 0x00000216F9103FB0>
Running agent Memory Agent (turn 1)
Tracing is disabled. Not creating span <agents.tracing.span_data.TurnSpanData object at 0x00000216FE68DD10>
No conversation_id available for request
Tracing is disabled. Not creating span <agents.tracing.span_data.GenerationSpanData object at 0x00000216FD443CB0>
Calling LLM
Received model response
Processing output item type=message class=ResponseOutputMessage
Resetting current trace
Turn 1: Got it! I've noted that your favorite color is **blue** and that you **love Python**.

Is there anything specific you'd like to explore with Python, or perhaps something you'd like to create?
Tracing is disabled. Not creating trace Agent workflow
Setting current trace

### Triage Agent OpenAI
Handoffs are the OpenAI Agents SDK's mechanism for one agent to delegate control to another. When an agent decides a request is better handled by a specialist, it hands off, and that specialist takes over the conversation. This is how we build multi-agent systems in the SDK: a triage agent that routes to a billing agent or a refunds agent, for instance.

In [11]:
from agents import Agent, Runner
import asyncio

# Enable detailed console logging
logger = logging.getLogger("openai.agents")
logger.handlers.clear()          # drop any handlers added by previous runs

spanish_agent = Agent(
    model=MODEL,
    name="spanish_agent",
    instructions="You only speak Spanish.",
)

english_agent = Agent(
    model=MODEL,
    name="english_agent",
    instructions="You only speak English",
)

triage_agent = Agent(
    model=MODEL,
    name="triage_agent",
    instructions="Handoff to the appropriate agent based on the language of the request.",
    handoffs=[spanish_agent, english_agent],
)

result = await Runner.run(triage_agent, input="Hola, ¿cómo estás?")
print(result.final_output)
# ¡Hola! Estoy bien, gracias por preguntar. ¿Y tú, cómo estás?
result = await Runner.run(triage_agent, input="Hello, How are you feeling today?")
print(result.final_output)

Tracing is disabled. Not creating trace Agent workflow
Setting current trace: no-op
Tracing is disabled. Not creating span <agents.tracing.span_data.TaskSpanData object at 0x00000216FE68D180>
Tracing is disabled. Not creating span <agents.tracing.span_data.AgentSpanData object at 0x00000216FD443F50>
Running agent triage_agent (turn 1)
Tracing is disabled. Not creating span <agents.tracing.span_data.TurnSpanData object at 0x00000216FE68D540>
No conversation_id available for request
Tracing is disabled. Not creating span <agents.tracing.span_data.GenerationSpanData object at 0x00000216FD443770>
Calling LLM
Received model response
Processing output item type=function_call class=ResponseFunctionToolCall
Tracing is disabled. Not creating span <agents.tracing.span_data.HandoffSpanData object at 0x00000216FE67C690>
Tracing is disabled. Not creating span <agents.tracing.span_data.AgentSpanData object at 0x00000216FD443F50>
Running agent spanish_agent (turn 2)
Tracing is disabled. Not creating 

### Parallel Agent Execution OpenAI
The parallel execution here is pure Python asyncio ; the Agents SDK isn't doing anything special for it.

In [12]:
import asyncio
from agents import Agent, Runner

# Enable detailed console logging
logger = logging.getLogger("openai.agents")
logger.handlers.clear()          # drop any handlers added by previous runs

research_agent = Agent(
    name="Researcher",
    instructions="Research the given topic and provide key facts.",
    model=MODEL
)

async def research_parallel(topics: list[str]) -> list[str]:
    """Run research agents in parallel for multiple topics."""
    tasks = [
        Runner.run(research_agent, f"Research: {topic}")
        for topic in topics
    ]
    results = await asyncio.gather(*tasks)
    return [r.final_output for r in results]

topics = ["Python history", "JavaScript history", "Rust history"]
results = await research_parallel(topics)
for topic, result in zip(topics, results):
    print(f"\n{topic}:\n{result[:200]}...")

Tracing is disabled. Not creating trace Agent workflow
Setting current trace: no-op
Tracing is disabled. Not creating span <agents.tracing.span_data.TaskSpanData object at 0x00000216FE68CCD0>
Tracing is disabled. Not creating span <agents.tracing.span_data.AgentSpanData object at 0x00000216F9103FB0>
Running agent Researcher (turn 1)
Tracing is disabled. Not creating span <agents.tracing.span_data.TurnSpanData object at 0x00000216FE71A670>
Tracing is disabled. Not creating trace Agent workflow
Setting current trace: no-op
Tracing is disabled. Not creating span <agents.tracing.span_data.TaskSpanData object at 0x00000216FE6F1860>
Tracing is disabled. Not creating span <agents.tracing.span_data.AgentSpanData object at 0x00000216FD443770>
Running agent Researcher (turn 1)
Tracing is disabled. Not creating span <agents.tracing.span_data.TurnSpanData object at 0x00000216FE6F17C0>
Tracing is disabled. Not creating trace Agent workflow
Setting current trace: no-op
Tracing is disabled. Not creat